# Chicago Crime & Socioeconomic Data — Download / Setup

This notebook makes sure both datasets are present in the `data/` folder and only downloads what is missing.

| # | Dataset | Kaggle slug |
|---|---------|-------------|
| 1 | Crimes – 2001 to Present | `utkarshx27/crimes-2001-to-present` |
| 2 | Socioeconomic Indicators in Chicago 2008–2012 | `umermjd11/socioeconomic-indicators-in-chicago-2008-2012` |

**Order of checks for each dataset:**
1. CSV already in `data/` → skip.
2. A local `.zip` already in `data/` (e.g. `Crimes_-_2001_to_Present.csv.zip`) → just extract it.
3. Otherwise download from Kaggle (needs a Kaggle API token).
4. If Kaggle isn't configured, fall back to the original source on the City of Chicago Data Portal (no login needed).

## 1. Install / import dependencies

In [1]:
# Uncomment on first run if packages are missing
# %pip install -q kaggle pandas requests tqdm

In [2]:
import os
import zipfile
from pathlib import Path

import pandas as pd
import requests
from tqdm.auto import tqdm

/Users/damithkayatech.com/Documents/MSc/BD Technologies Assignment/PRAC 1/Chicago_Crime_Analysis_Project/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configuration

In [3]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = {
    "crimes": {
        "kaggle_slug": "utkarshx27/crimes-2001-to-present",
        # glob patterns used to detect whether the CSV already exists
        "csv_patterns": ["Crimes*2001*Present*.csv", "*crimes*.csv"],
        "zip_patterns": ["Crimes*2001*Present*.zip", "crimes-2001-to-present.zip"],
        # Fallback: City of Chicago Data Portal (the original source of the Kaggle data)
        "fallback_url": "https://data.cityofchicago.org/api/views/ijzp-q8t2/rows.csv?accessType=DOWNLOAD",
        "fallback_name": "Crimes_-_2001_to_Present.csv",
    },
    "socioeconomic": {
        "kaggle_slug": "umermjd11/socioeconomic-indicators-in-chicago-2008-2012",
        "csv_patterns": ["*ocioeconomic*.csv", "Census_Data*.csv"],
        "zip_patterns": ["*ocioeconomic*.zip"],
        "fallback_url": "https://data.cityofchicago.org/api/views/kn9c-c2s2/rows.csv?accessType=DOWNLOAD",
        "fallback_name": "Census_Data_-_Selected_socioeconomic_indicators_in_Chicago__2008___2012.csv",
    },
}

## 3. Helper functions

In [4]:
def find_files(patterns, folder=DATA_DIR):
    """Return files in `folder` (recursively) matching any glob pattern."""
    hits = []
    for p in patterns:
        hits.extend(folder.rglob(p))
    return sorted(set(hits))


def extract_zip(zip_path, dest=DATA_DIR):
    print(f"  Extracting {zip_path.name} ...")
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(dest)
    print("  Done.")


def kaggle_available():
    """True if a Kaggle token is configured (kaggle.json or env vars)."""
    token = Path.home() / ".kaggle" / "kaggle.json"
    return token.exists() or ("KAGGLE_USERNAME" in os.environ and "KAGGLE_KEY" in os.environ)


def download_from_kaggle(slug, dest=DATA_DIR):
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi()
    api.authenticate()
    print(f"  Downloading {slug} from Kaggle ...")
    api.dataset_download_files(slug, path=str(dest), unzip=True, quiet=False)
    print("  Done.")


def download_url(url, out_path, chunk=1 << 20):
    """Stream a (possibly very large) file to disk with a progress bar."""
    tmp = out_path.with_suffix(out_path.suffix + ".part")
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0)) or None
        with open(tmp, "wb") as f, tqdm(total=total, unit="B", unit_scale=True,
                                        desc=out_path.name) as bar:
            for block in r.iter_content(chunk_size=chunk):
                f.write(block)
                bar.update(len(block))
    tmp.rename(out_path)  # only rename once the download is complete


def ensure_dataset(name, cfg):
    print(f"[{name}]")

    # 1) CSV already present?
    csvs = find_files(cfg["csv_patterns"])
    if csvs:
        print(f"  Found: {[str(c) for c in csvs]} -> skipping download.")
        return csvs[0]

    # 2) Local zip present (e.g. the attached Crimes_-_2001_to_Present.csv.zip)?
    zips = find_files(cfg["zip_patterns"])
    if zips:
        extract_zip(zips[0])
        csvs = find_files(cfg["csv_patterns"])
        if csvs:
            return csvs[0]

    # 3) Kaggle
    if kaggle_available():
        try:
            download_from_kaggle(cfg["kaggle_slug"])
            csvs = find_files(cfg["csv_patterns"])
            if csvs:
                return csvs[0]
        except Exception as e:
            print(f"  Kaggle download failed: {e}")
    else:
        print("  No Kaggle token found (~/.kaggle/kaggle.json or KAGGLE_USERNAME/KAGGLE_KEY).")

    # 4) Fallback: City of Chicago Data Portal
    out = DATA_DIR / cfg["fallback_name"]
    print(f"  Falling back to Chicago Data Portal -> {out}")
    download_url(cfg["fallback_url"], out)
    return out

## 4. (Optional) Kaggle credentials

Get a token from **kaggle.com → Settings → API → Create New Token**, then either place `kaggle.json` in `~/.kaggle/` (Windows: `C:\Users\<you>\.kaggle\`) or set the variables below. Skip this cell if you prefer the Chicago Data Portal fallback.

In [5]:
# os.environ["KAGGLE_USERNAME"] = "your_username"
# os.environ["KAGGLE_KEY"] = "your_api_key"
print("Kaggle configured:", kaggle_available())

Kaggle configured: False


## 5. Ensure both datasets are available

In [6]:
paths = {name: ensure_dataset(name, cfg) for name, cfg in DATASETS.items()}
paths

[crimes]
  Found: ['data/Crimes_-_2001_to_Present.csv'] -> skipping download.
[socioeconomic]
  Found: ['data/Census_Data_-_Selected_socioeconomic_indicators_in_Chicago__2008___2012_20240509.csv'] -> skipping download.


{'crimes': PosixPath('data/Crimes_-_2001_to_Present.csv'),
 'socioeconomic': PosixPath('data/Census_Data_-_Selected_socioeconomic_indicators_in_Chicago__2008___2012_20240509.csv')}

## 6. Quick verification

The crimes file is large (~1.8 GB, 8M+ rows), so only a sample is loaded here.

In [7]:
crimes_sample = pd.read_csv(paths["crimes"], nrows=5)
print("Crimes file size: %.1f MB" % (paths["crimes"].stat().st_size / 1e6))
crimes_sample

Crimes file size: 1837.9 MB


,ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,...,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Year,Updated On,Latitude,Longitude,Location
0,10224738,HY411648,09/05/2015 01:30:00 PM,043XX S WOOD ST,486,BATTERY,DOMESTIC BATTERY SIMPLE,RESIDENCE,False,True,...,12,61,08B,1165074.0,1875917.0,2015,02/10/2018 03:50:01 PM,41.815117,-87.670000,"(41.815117282, -87.669999562)"
1,10224739,HY411615,09/04/2015 11:30:00 AM,008XX N CENTRAL AVE,870,THEFT,POCKET-PICKING,CTA BUS,False,False,...,29,25,06,1138875.0,1904869.0,2015,02/10/2018 03:50:01 PM,41.895080,-87.765400,"(41.895080471, -87.765400451)"
2,11646166,JC213529,09/01/2018 12:01:00 AM,082XX S INGLESIDE AVE,810,THEFT,OVER $500,RESIDENCE,False,True,...,8,44,06,NaN,NaN,2018,04/06/2019 04:04:43 PM,NaN,NaN,NaN
3,10224740,HY411595,09/05/2015 12:45:00 PM,035XX W BARRY AVE,2023,NARCOTICS,POSS: HEROIN(BRN/TAN),SIDEWALK,True,False,...,35,21,18,1152037.0,1920384.0,2015,02/10/2018 03:50:01 PM,41.937406,-87.716650,"(41.937405765, -87.716649687)"
4,10224741,HY411610,09/05/2015 01:00:00 PM,0000X N LARAMIE AVE,560,ASSAULT,SIMPLE,APARTMENT,False,True,...,28,25,08A,1141706.0,1900086.0,2015,02/10/2018 03:50:01 PM,41.881903,-87.755121,"(41.881903443, -87.755121152)"


In [8]:
socio = pd.read_csv(paths["socioeconomic"])
print(socio.shape)
socio.head()

(78, 9)


,Community Area Number,COMMUNITY AREA NAME,PERCENT OF HOUSING CROWDED,PERCENT HOUSEHOLDS BELOW POVERTY,PERCENT AGED 16+ UNEMPLOYED,PERCENT AGED 25+ WITHOUT HIGH SCHOOL DIPLOMA,PERCENT AGED UNDER 18 OR OVER 64,PER CAPITA INCOME,HARDSHIP INDEX
0,1.0,Rogers Park,7.7,23.6,8.7,18.2,27.5,23939,39.0
1,2.0,West Ridge,7.8,17.2,8.8,20.8,38.5,23040,46.0
2,3.0,Uptown,3.8,24.0,8.9,11.8,22.2,35787,20.0
3,4.0,Lincoln Square,3.4,10.9,8.2,13.4,25.5,37524,17.0
4,5.0,North Center,0.3,7.5,5.2,4.5,26.2,57123,6.0


### Loading the full crimes data efficiently (optional)

Reading only the columns you need keeps memory usage manageable.

In [9]:
# usecols = ["ID", "Date", "Primary Type", "Arrest", "Domestic",
#            "Community Area", "Year", "Latitude", "Longitude"]
# crimes = pd.read_csv(paths["crimes"], usecols=usecols, low_memory=False)
# crimes["Date"] = pd.to_datetime(crimes["Date"], format="%m/%d/%Y %I:%M:%S %p")
# crimes.info(memory_usage="deep")